# Clinical Trial Designer — GRPO Training Notebook

**OpenEnv Hackathon | Meta PyTorch × Scaler School of Technology**

This notebook trains an LLM to design clinical trials using GRPO (Group Relative Policy Optimization) from HuggingFace TRL + Unsloth for efficient LoRA fine-tuning.

**Requirements:**
- Colab with T4 or better GPU (free tier works with 4-bit quantization)
- Running OpenEnv environment server (Docker or HF Space)

**What this notebook does:**
1. Installs dependencies (TRL, Unsloth, vLLM)
2. Connects to the Clinical Trial Designer environment
3. Configures GRPO with LoRA
4. Trains the agent with reward feedback from the environment
5. Evaluates trained vs base model
6. Saves checkpoint to HuggingFace Hub

## 1. Install Dependencies

In [ ]:
%%capture
# Install Unsloth for fast LoRA training (2x speed, 50% less VRAM)
!pip install unsloth
# Install TRL for GRPO trainer
!pip install trl>=0.29.0
# Install other dependencies
!pip install requests scipy numpy datasets

## 2. Environment Connection

The Clinical Trial Designer environment runs as a FastAPI server (OpenEnv compatible). It exposes `/reset`, `/step`, `/state`, and `/ping` endpoints.

**Option A:** Run the Docker container locally and use ngrok to expose it to Colab.  
**Option B:** Deploy to HuggingFace Spaces and use the Space URL.  
**Option C:** Run locally with port forwarding.

In [ ]:
import requests
import json

# Set your environment URL here
ENV_URL = "http://localhost:8000"  # Change to your HF Space URL or ngrok URL

def env_reset(scenario_id=None, tier=0):
    """Reset environment and get initial observation."""
    payload = {"tier": tier}
    if scenario_id:
        payload["scenario_id"] = scenario_id
    resp = requests.post(f"{ENV_URL}/reset", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_step(action_type, parameters=None):
    """Take an action in the environment."""
    payload = {"action_type": action_type, "parameters": parameters or {}}
    resp = requests.post(f"{ENV_URL}/step", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_ping():
    """Health check."""
    resp = requests.get(f"{ENV_URL}/ping", timeout=10)
    return resp.json()

# Test connection
try:
    print("Ping:", env_ping())
    obs = env_reset(tier=0)
    print("Reset OK. Scenario:", obs.get("scenario_id", "unknown"))
    print("Available actions:", obs.get("available_actions", [])[:5], "...")
except Exception as e:
    print(f"Cannot connect to environment at {ENV_URL}: {e}")
    print("Start the Docker container or set ENV_URL to your HF Space.")

## 3. Load Model with Unsloth

We use Unsloth for 2× faster LoRA fine-tuning with lower VRAM usage. The model is loaded in 4-bit quantization to fit on a T4 GPU (16GB).

In [ ]:
from unsloth import FastLanguageModel

# Load base model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=4096,
    dtype=None,            # Auto-detect
    load_in_4bit=True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # LoRA rank
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

## 4. Define Reward Function

The reward function runs a full episode in the environment for each completion. The agent's response is parsed as a JSON action, stepped through the environment, and the cumulative reward is returned.

**Reward range:** -3 to +14 (high variance for GRPO advantage computation).

In [ ]:
import re

SYSTEM_PROMPT = """You are a clinical trial designer. You receive a disease scenario 
and must design a complete Phase I/II clinical trial.

Your goal: design a trial that detects the drug's true effect and identifies 
the right patient population, within budget and FDA regulations.

Available actions (respond with JSON):
PHASE I: run_dose_escalation, observe_safety_signal, estimate_effect_size
PHASE II: set_primary_endpoint, set_sample_size, set_inclusion_criteria,
    set_exclusion_criteria, set_dosing_schedule, set_control_arm,
    set_randomization_ratio, set_blinding
REGULATORY: submit_to_fda_review, request_protocol_amendment
MONITORING: run_interim_analysis, modify_sample_size, add_biomarker_stratification
ANALYSIS: run_primary_analysis
CONCLUSION: synthesize_conclusion

Follow Phase I -> Phase II -> FDA -> Monitoring -> Analysis -> Conclusion.
Respond: {"action_type": "<action>", "parameters": {...}}"""


def parse_action(text):
    """Extract JSON action from model output."""
    # Try to find JSON in the response
    match = re.search(r'\{[^{}]*"action_type"[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"action_type": "synthesize_conclusion", "parameters": {}}


def run_episode(model_response, tier=0):
    """Run a full episode from a model's multi-step plan.
    
    For simplicity in the Colab demo, we treat each GRPO completion
    as a single action. In the full train.py, the agent runs
    multi-turn episodes with the environment.
    """
    try:
        obs = env_reset(tier=tier)
        total_reward = 0.0
        done = False
        steps = 0
        max_steps = 100
        
        # Parse all actions from the model's response
        actions = []
        for line in model_response.split('\n'):
            parsed = parse_action(line)
            if parsed["action_type"] != "synthesize_conclusion" or not actions:
                actions.append(parsed)
        
        # If model gave a single action, just use it
        if not actions:
            actions = [parse_action(model_response)]
        
        for action in actions:
            if done or steps >= max_steps:
                break
            result = env_step(action["action_type"], action.get("parameters", {}))
            total_reward += result.get("reward", 0.0)
            done = result.get("done", False)
            steps += 1
        
        return total_reward
    except Exception as e:
        print(f"Episode error: {e}")
        return -2.0  # Timeout-equivalent penalty


def reward_func(completions, **kwargs):
    """GRPO reward function. Takes list of completions, returns list of rewards."""
    rewards = []
    for completion in completions:
        # Extract text from completion
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        reward = run_episode(text, tier=0)
        rewards.append(reward)
    return rewards

print("Reward function defined. Range: -3 to +14")

## 5. Prepare Training Dataset

GRPO needs prompts to generate completions for. Each prompt describes a clinical trial scenario. The model generates an action plan, which is scored by the environment.

In [ ]:
from datasets import Dataset

# Scenario prompts — the agent sees these, not the hidden ground truth
SCENARIO_PROMPTS = [
    {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": """Scenario: solid_tumor_chemo
Disease: Non-small cell lung cancer (NSCLC)
Drug: Novel EGFR tyrosine kinase inhibitor (3rd generation)
Budget: $2,500,000 | Time: 180 days
Challenge: Design a Phase I/II trial for this drug.

Plan your complete trial design step by step. For each step, provide a JSON action."""}
        ],
    },
    {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": """Scenario: autoimmune_biologic
Disease: Rheumatoid arthritis (RA)
Drug: Novel anti-IL-6 biologic
Budget: $1,800,000 | Time: 150 days
Challenge: The drug may have a U-shaped dose-response. Design a trial to find the optimal dose.

Plan your complete trial design step by step. For each step, provide a JSON action."""}
        ],
    },
    {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": """Scenario: cns_depression
Disease: Treatment-resistant depression (TRD)
Drug: Novel rapid-acting antidepressant
Budget: $3,000,000 | Time: 210 days
Challenge: High placebo response rate may mask the true drug effect. Design carefully.

Plan your complete trial design step by step. For each step, provide a JSON action."""}
        ],
    },
    {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": """Scenario: rare_disease_orphan
Disease: Morquio A syndrome (rare pediatric metabolic disorder)
Drug: Enzyme replacement therapy
Budget: $1,200,000 | Time: 240 days
Challenge: Very small patient population (max ~50). Adaptive design required.

Plan your complete trial design step by step. For each step, provide a JSON action."""}
        ],
    },
]

# Create dataset with repeated prompts for training
train_prompts = SCENARIO_PROMPTS * 50  # 200 prompts total (50 per scenario)
train_dataset = Dataset.from_list(train_prompts)

print(f"Training dataset: {len(train_dataset)} prompts ({len(SCENARIO_PROMPTS)} scenarios × 50 repeats)")

## 6. Configure and Run GRPO Training

GRPO generates `num_generations` completions per prompt, scores them with the reward function, and updates the policy to favor higher-reward completions.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# GRPO configuration
training_args = GRPOConfig(
    output_dir="checkpoints/grpo_clinical_trial",
    
    # GRPO-specific
    num_generations=8,                 # 8 rollouts per prompt
    max_completion_length=512,         # Max tokens per action plan
    temperature=0.7,
    
    # Training
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=100,                     # Reduce for Colab demo; use 500 for full training
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Logging
    logging_steps=1,
    save_steps=25,
    report_to="none",
    
    # Precision
    bf16=True,
)

# Create trainer
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    reward_funcs=[reward_func],
)

print("GRPO Trainer configured.")
print(f"  Steps: {training_args.max_steps}")
print(f"  Rollouts per step: {training_args.num_generations}")
print(f"  Total episodes: ~{training_args.max_steps * training_args.num_generations}")

In [ ]:
# Run training!
print("Starting GRPO training...")
print("Monitor reward values in the training log below.")
print("Expected: rewards should trend upward over 100+ steps.")
print("="*60)

trainer.train()

print("="*60)
print("Training complete!")

## 7. Evaluate Trained Model

Compare the trained model against a random baseline to demonstrate learning.

In [ ]:
import random

def evaluate_policy(policy_fn, n_episodes=20, tier=0):
    """Evaluate a policy over N episodes."""
    results = []
    for i in range(n_episodes):
        scenario = random.choice(["solid_tumor_chemo", "autoimmune_biologic",
                                   "cns_depression", "rare_disease_orphan"])
        obs = env_reset(scenario_id=scenario, tier=tier)
        total_reward = 0.0
        done = False
        steps = 0
        
        while not done and steps < 100:
            action = policy_fn(obs)
            result = env_step(action["action_type"], action.get("parameters", {}))
            total_reward += result.get("reward", 0.0)
            done = result.get("done", False)
            obs = result.get("observation", obs)
            steps += 1
        
        results.append({
            "scenario": scenario,
            "reward": total_reward,
            "steps": steps,
            "success": result.get("success", False),
        })
    return results


def random_policy(obs):
    """Random action from available actions."""
    actions = obs.get("available_actions", ["synthesize_conclusion"])
    return {"action_type": random.choice(actions), "parameters": {}}


def trained_policy(obs):
    """Use trained model to select action."""
    prompt = f"""Current observation: {json.dumps(obs, default=str)[:500]}
Select your next action as JSON: {{"action_type": "...", "parameters": {{...}}}}"""
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
    outputs = model.generate(inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return parse_action(response)


print("Evaluating random baseline (20 episodes)...")
random_results = evaluate_policy(random_policy, n_episodes=20)

print("Evaluating trained model (20 episodes)...")
trained_results = evaluate_policy(trained_policy, n_episodes=20)

# Compare
random_avg = sum(r["reward"] for r in random_results) / len(random_results)
trained_avg = sum(r["reward"] for r in trained_results) / len(trained_results)
random_success = sum(r["success"] for r in random_results) / len(random_results)
trained_success = sum(r["success"] for r in trained_results) / len(trained_results)

print(f"\n{'='*50}")
print(f"{'Metric':<25} {'Random':>10} {'Trained':>10}")
print(f"{'='*50}")
print(f"{'Avg Reward':<25} {random_avg:>10.2f} {trained_avg:>10.2f}")
print(f"{'Success Rate':<25} {random_success:>10.1%} {trained_success:>10.1%}")
print(f"{'Improvement':<25} {'':>10} {trained_avg - random_avg:>+10.2f}")
print(f"{'='*50}")

## 8. Save Checkpoint to HuggingFace Hub

Upload the trained LoRA adapter to HuggingFace for deployment and demo.

In [ ]:
from huggingface_hub import notebook_login

# Login to HuggingFace (required for push)
notebook_login()

In [ ]:
# Save and push to Hub
REPO_ID = "Roopalgn/clinical-trial-designer-grpo"

# Save locally
model.save_pretrained("checkpoints/grpo_clinical_trial/final")
tokenizer.save_pretrained("checkpoints/grpo_clinical_trial/final")

# Push to HF Hub
model.push_to_hub(REPO_ID, commit_message="GRPO trained clinical trial designer")
tokenizer.push_to_hub(REPO_ID, commit_message="GRPO trained clinical trial designer")

print(f"Model uploaded to: https://huggingface.co/{REPO_ID}")

## 9. Generate Reward Curve Plot

Quick visualization of training progress for the pitch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract rewards from trainer logs
if hasattr(trainer, 'state') and trainer.state.log_history:
    rewards = [log.get('reward', None) for log in trainer.state.log_history if 'reward' in log]
    
    if rewards:
        episodes = range(1, len(rewards) + 1)
        
        # Rolling average
        window = min(20, len(rewards) // 3 + 1)
        rolling_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
        
        # Trend line
        z = np.polyfit(range(len(rewards)), rewards, 1)
        trend = np.poly1d(z)
        
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.scatter(episodes, rewards, alpha=0.3, color='#4a90d9', s=20, label='Per-episode')
        ax.plot(range(window, len(rewards) + 1), rolling_avg, color='#e63946',
                linewidth=2, label=f'Rolling avg (w={window})')
        ax.plot(episodes, trend(range(len(rewards))), '--', color='#2d6a4f',
                linewidth=1.5, label=f'Trend (slope={z[0]:.3f})')
        
        ax.set_xlabel('Episode', fontsize=12)
        ax.set_ylabel('Total Reward', fontsize=12)
        ax.set_title('Training Reward Curve — Clinical Trial Designer', fontsize=14)
        ax.legend(loc='upper left')
        ax.grid(True, alpha=0.3)
        
        # Annotations
        ax.annotate(
            f'Best: {max(rewards):.1f}\nFinal avg: {np.mean(rewards[-20:]):.1f}\nSlope: {z[0]:.3f}',
            xy=(0.98, 0.02), xycoords='axes fraction',
            ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
        )
        
        plt.tight_layout()
        plt.savefig('results/reward_curve.png', dpi=150)
        plt.show()
        print(f"Saved to results/reward_curve.png")
    else:
        print("No reward data found in training logs.")
else:
    print("Training logs not available. Run training first.")

## Summary

This notebook demonstrates:
1. **Environment Innovation** — a clinical trial simulator with hidden ground truth, multi-layer verification, and 5-tier curriculum
2. **Reward Design** — 8 per-step + 7 terminal components with potential-based shaping
3. **Training** — GRPO with Unsloth/TRL producing measurable improvement
4. **Proof of Learning** — reward curves trending upward, trained model beating random baseline

For full documentation, see the [GitHub repository](https://github.com/Roopalgn/openenv-clinical-trial).